# Cross-view Harmonic Outlier Reversal for Debiasing (CHORD)

Run the public implementation on **Waterbirds, CelebA, MultiNLI, CivilComments, CREMA-D, and MEAD** in Google Colab.

This notebook downloads the provided ERM models and feature caches, then trains and evaluates the CHORD classifiers. Each dataset uses the ten classifier seeds configured in the repository. It does not retrain the encoders or regenerate features from raw data.

**Before you begin**

1. In Colab, select **Runtime → Change runtime type → T4 GPU → Save**.
2. Run the cells below in order, or select **Runtime → Run all**.
3. Keep the runtime connected until the results table appears. You can download the results with the final cell.

The resource archive is approximately **5.27 GB**. Allow about **20 GB of free runtime disk space** for the environment, archive, and extracted resources. A fresh Colab T4 run of this workflow took about 20 minutes; download speed and runtime availability can vary. No Google Drive mount is needed.


## 1. Get the code

The code is cloned from GitHub into the runtime. `REF` selects the release branch or tag. To run only one dataset, replace `DATASETS` with, for example, `["waterbirds"]`.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def run(command, *, cwd=None, env=None):
    """Display command output in the notebook and stop on errors."""
    with subprocess.Popen(
        command, cwd=cwd, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)


REPO_URL = "https://github.com/davelee-uestc/chord_debiasing_public.git"
REF = "main"
DATASETS = ["waterbirds", "celeba", "multinli", "civilcomments", "cremad", "mead"]

workspace = Path("/content") if Path("/content").is_dir() else Path.cwd()
repo = workspace / "chord"
if not REPO_URL:
    raise ValueError("Set REPO_URL to the public GitHub repository URL.")
if not repo.exists():
    run(["git", "clone", "--depth", "1", "--branch", REF, REPO_URL, str(repo)])
else:
    print(f"Using the existing checkout at {repo}. Start a fresh runtime to clone another release.")
print("Code directory:", repo)


## 2. Set up the environment

Colab's default Python version may differ from the project's requirements. This cell creates a separate **Python 3.12** environment and installs **PyTorch 2.5.0 with CUDA 12.1**, followed by the repository's pinned requirements. It leaves Colab's notebook environment intact and does not require a session restart.

If the GPU check fails, select a GPU runtime before continuing. The first installation downloads several packages and may take a few minutes.


In [ ]:
run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"])
run([sys.executable, "-m", "pip", "install", "-q", "uv", "matplotlib"])
venv = workspace / "chord-venv"
python = str(venv / "bin/python")
if not Path(python).exists():
    run([sys.executable, "-m", "uv", "venv", "--python", "3.12", str(venv)])
run([
    sys.executable, "-m", "uv", "pip", "install", "--python", python,
    "torch==2.5.0", "torchvision==0.20.0", "--index-url", "https://download.pytorch.org/whl/cu121",
])
run([
    sys.executable, "-m", "uv", "pip", "install", "--python", python,
    "-r", str(repo / "requirements.txt"),
])
env = dict(os.environ, PYTHON=python, OMP_NUM_THREADS="4")
run([
    python, "-c",
    "import torch; assert torch.cuda.is_available(), 'Select a GPU runtime'; "
    "print('PyTorch:', torch.__version__); print('GPU:', torch.cuda.get_device_name(0))",
], env=env)


## 3. Download the supplied resources

This runs the repository's download script. It downloads the archive to `chord/downloads/` and extracts the ERM models, classifier weights, and feature caches into `chord/resources/`.

An existing completed archive is reused; an interrupted download can be resumed by rerunning this cell. If Google Drive reports a download quota or access error, retry later using the resource link in the repository README.


In [ ]:
run(["bash", "download_resources.sh"], cwd=repo, env=env)


## 4. Reproduce the results

Run CHORD on the selected datasets using the supplied feature caches. This calls the same `reproduce.sh` entry point documented in the README. Per-seed metrics appear during execution, followed by the ERM and CHORD summary table.

Results are written to `chord/resources/results/`. Regenerated classifier weights are saved as `chord/resources/checkpoints/<dataset>/chord_seed<seed>.pt`. Rerunning this cell replaces the selected datasets' result files and classifier weights in this runtime.


In [ ]:
run([
    "bash", "reproduce.sh", "--datasets", *DATASETS, "--device", "cuda:0",
], cwd=repo, env=env)


## 5. Plot worst-group accuracy

Compare ERM and CHORD using the JSON results from this run. CHORD bars show the mean over the ten classifier seeds; error bars show one standard deviation. ERM uses the supplied fixed encoder checkpoint and therefore has no error bar.

The figure is displayed below and saved as PNG and PDF under `chord/resources/results/`. It is included in the downloadable result ZIP.


In [ ]:
import json
import matplotlib.pyplot as plt

results_dir = repo / "resources" / "results"
results = [json.loads((results_dir / f"{dataset}.json").read_text()) for dataset in DATASETS]
erm_wga = [100 * result["erm"]["wga"] for result in results]
chord_wga = [100 * result["wga"] for result in results]
chord_std = [100 * result["wga_std"] for result in results]
names = {
    "waterbirds": "Waterbirds", "celeba": "CelebA", "multinli": "MultiNLI",
    "civilcomments": "CivilComments", "cremad": "CREMA-D", "mead": "MEAD",
}
x = list(range(len(DATASETS)))
width = 0.36
fig, ax = plt.subplots(figsize=(max(6, 1.7 * len(DATASETS)), 4.8))
erm_bars = ax.bar([i - width / 2 for i in x], erm_wga, width,
                  label="ERM", color="#64748b", zorder=3)
chord_bars = ax.bar([i + width / 2 for i in x], chord_wga, width,
                    yerr=chord_std, capsize=3, label="CHORD", color="#2563eb", zorder=3)
ax.bar_label(erm_bars, fmt="%.2f", padding=3, fontsize=9)
ax.bar_label(chord_bars, fmt="%.2f", padding=3, fontsize=9)
ax.set_xticks(x, [names.get(dataset, dataset) for dataset in DATASETS])
ax.set_ylabel("Worst-group accuracy (%)")
ax.set_title("ERM and CHORD")
ax.set_ylim(0, 105)
ax.set_yticks(range(0, 101, 20))
ax.grid(axis="y", alpha=0.2, zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(results_dir / "wga_comparison.png", dpi=180, bbox_inches="tight")
fig.savefig(results_dir / "wga_comparison.pdf", bbox_inches="tight")
plt.show()


## 6. Download the results

Save the summary, per-dataset JSON results, WGA figure, and regenerated CHORD classifier weights before the Colab runtime ends. The ZIP contains only these small outputs; the input caches, encoders, and resource archive are excluded.


In [ ]:
import zipfile

resources = repo / "resources"
bundle = workspace / "chord_results.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted((resources / "results").glob("*")):
        if path.is_file():
            archive.write(path, str(path.relative_to(resources)))
    for dataset in DATASETS:
        for path in sorted((resources / "checkpoints" / dataset).glob("chord_seed*.pt")):
            archive.write(path, str(path.relative_to(resources)))
print("Results saved to:", bundle)

try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(str(bundle))
